In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import numpy as np
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

In [ ]:
outlined_img = cv2.imread("manual_data/manual_outlines.png")
outlines = outlined_img - np.mean(outlined_img, axis=2, dtype=np.float32)[:, :, None]
# cv2.imwrite("outlines.png", outlines)

In [ ]:
# Convert to grayscale
gray = cv2.cvtColor(outlines.astype(np.uint8), cv2.COLOR_BGR2GRAY)

# Since rings are solid color, simple threshold works
_, binary = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)

# Find ALL contours including holes (important for rings)
contours, hierarchy = cv2.findContours(
    binary,
    cv2.RETR_TREE,          # keeps parent-child relationships
    cv2.CHAIN_APPROX_SIMPLE
)

print("Total contours:", len(contours))

Total contours: 764


In [ ]:
outer_rings = []

for i, h in enumerate(hierarchy[0]):
    parent = h[3]
    if parent == -1:
        outer_rings.append(contours[i])

print("Number of rings:", len(outer_rings))

Number of rings: 361


In [ ]:
import cv2
import numpy as np
from typing import Tuple

# Copy original image for overlay
overlay = np.zeros((8192, 8192, 3), dtype=np.uint8)

past_colour_codes : set[Tuple[int, int, int]] = {(0, 0, 0)} # Use (0, 0, 0) as the background

# Fill each ring with unique colour
for contour in outer_rings:
    while True:
        colour_code = (
            int(np.random.randint(0, 255, dtype=np.uint8)),
            int(np.random.randint(0, 255, dtype=np.uint8)),
            int(np.random.randint(0, 255, dtype=np.uint8))
        )

        if colour_code not in past_colour_codes:
            past_colour_codes.add(colour_code)
            break

    cv2.drawContours(overlay, [contour], -1, list(colour_code), thickness=cv2.FILLED)

In [ ]:
j, i= np.meshgrid(
    np.arange(8192, dtype=np.int32),
    np.arange(8192, dtype=np.int32)
)

i = np.flip(i)
# j = np.flip(j)

i_flat = i.ravel()
j_flat = j.ravel()

r_flat = overlay[..., 0].ravel()
g_flat = overlay[..., 1].ravel()
b_flat = overlay[..., 2].ravel()

detection_codes = (
    r_flat.astype(np.uint32)
    | (g_flat.astype(np.uint32) << 8)
    | (b_flat.astype(np.uint32) << 16)
)

df = pl.DataFrame(
    {
        "i": i_flat,
        "j": j_flat,
        "manual_detect_boulder_id": detection_codes,
    }
).with_columns(
    pl.lit("posx").alias("face")
)

df.filter(pl.col("manual_detect_boulder_id") != 0).write_parquet(".database/manual_detection_database.parquet")